## Libraries and Data

In [1]:
import spacy
import pandas as pd
import re
from pathlib import Path
from spacy.matcher import Matcher

# Carregamento direto dos arquivos
ROOT = Path('../../data/raw')
data = pd.read_csv(ROOT / 'cases.csv')
metadata = pd.read_csv(ROOT / 'metadata.csv')

# Merge e seleção do primeiro texto clínico
full_data = pd.merge(data, metadata)
full_data = full_data[['case_text', 'gender', 'case_id', 'major_mesh_terms', 'mesh_terms']]
text = full_data['case_text'][0]

# Inicialização do spaCy
nlp = spacy.load('en_core_web_sm')
processed_text = nlp(text)

## Captura de Números e Unidades (Regex)

In [2]:
measurement_pattern = re.compile(r'(\d+(?:,\d+)?(?:\.\d+)?)\s*(cm|mm|ng/ml|iu/ml|mg)')

measurements = []
for match in measurement_pattern.finditer(text):
    value, unit = match.group(1), match.group(2)
    measurements.append({
        'node_type': 'ExamResult',
        'value': value,
        'unit': unit,
        'label_original': match.group(0),
        'label_normalizado': f"{value} {unit}",
        'token_start': -1,
        'token_end': -1,
        'span_start': match.start(),
        'span_end': match.end()
    })

measurements_df = pd.DataFrame(measurements)
print("Resultados de Exames Extraídos:")
display(measurements_df)

Resultados de Exames Extraídos:


,node_type,value,unit,label_original,label_normalizado,token_start,token_end,span_start,span_end
0,ExamResult,6,cm,6cm,6 cm,-1,-1,337,340
1,ExamResult,6,cm,6cm,6 cm,-1,-1,516,519
2,ExamResult,9,cm,9cm,9 cm,-1,-1,522,525
3,ExamResult,"12,476.5",ng/ml,"12,476.5ng/ml","12,476.5 ng/ml",-1,-1,777,790
4,ExamResult,6,iu/ml,6iu/ml,6 iu/ml,-1,-1,837,843
5,ExamResult,9.5,cm,9.5cm,9.5 cm,-1,-1,2070,2075
6,ExamResult,4.5,cm,4.5cm,4.5 cm,-1,-1,2078,2083
7,ExamResult,2.0,cm,2.0cm,2.0 cm,-1,-1,2086,2091


## Extração de Conceitos Médicos (Matcher Gramatical)

In [3]:
matcher = Matcher(nlp.vocab)
pattern_clinical = [{"POS": {"IN": ["ADJ", "NOUN"]}}, {"POS": "NOUN"}]
matcher.add("CLINICAL_TERM", [pattern_clinical])

extracted_entities = [
    {
        'node_type': 'MedicalConcept',
        'label_original': processed_text[start:end].text,
        'label_normalizado': processed_text[start:end].lemma_,
        'token_start': start,
        'token_end': end,
        'span_start': processed_text[start:end].start_char,
        'span_end': processed_text[start:end].end_char,
        'value': None,
        'unit': None
    }
    for _, start, end in matcher(processed_text)
]

nodes_df = pd.DataFrame(extracted_entities).drop_duplicates(subset=['label_normalizado']).reset_index(drop=True)
display(nodes_df.head(15))

,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,MedicalConcept,old woman,old woman,5,7,10,19,None,None
1,MedicalConcept,day history,day history,12,14,39,50,None,None
2,MedicalConcept,right flank,right flank,15,17,54,65,None,None
3,MedicalConcept,abdominal pain,abdominal pain,20,22,85,99,None,None
4,MedicalConcept,medication history,medication history,34,36,170,188,None,None
5,MedicalConcept,physical examination,physical examination,43,45,229,249,None,None
6,MedicalConcept,computed tomography,computed tomography,52,54,300,319,None,None
7,MedicalConcept,cystic lesion,cystic lesion,59,61,341,354,None,None
8,MedicalConcept,pancreatic echotexture,pancreatic echotexture,86,88,472,494,None,None
9,MedicalConcept,internal septations,internal septation,101,103,543,562,None,None


## Extração de Arestas Semânticas por Dependência Sintática

In [4]:
dependency_edges = []
edge_id_counter = 1

verb_relations = {
    ("present", "have", "experience"): "HAS_SYMPTOM",
    ("undergo", "perform", "do", "plan"): "UNDERWENT_EXAM",
    ("reveal", "demonstrate", "suggest", "show", "consist"): "SUPPORTS",
    ("treat", "resect", "discharge", "perform"): "TREATED_BY"
}

for sent in processed_text.sents:
    nodes_in_sent = [row for _, row in nodes_df.iterrows() if sent.start <= row['token_start'] < sent.end]
    
    if len(nodes_in_sent) >= 2:
        for token in sent:
            if token.pos_ == "VERB":
                verbo_lema = token.lemma_.lower()
                relacao = "RELATED_TO"
                
                for verbs, rel in verb_relations.items():
                    if verbo_lema in verbs:
                        relacao = rel
                        break

                for i in range(len(nodes_in_sent) - 1):
                    dependency_edges.append({
                        'edge_id': f"E_DEP_{edge_id_counter:03d}",
                        'source_label': nodes_in_sent[i]['label_normalizado'],
                        'target_label': nodes_in_sent[i+1]['label_normalizado'],
                        'relation': relacao,
                        'trigger_verb': verbo_lema,
                        'sentence': sent.text.strip()
                    })
                    edge_id_counter += 1

edges_dep_df = pd.DataFrame(dependency_edges).drop_duplicates(subset=['source_label', 'target_label', 'relation'])
print(f"Total de arestas por dependência sintática: {len(edges_dep_df)}")
display(edges_dep_df.head(10))

Total de arestas por dependência sintática: 41


,edge_id,source_label,target_label,relation,trigger_verb,sentence
0,E_DEP_001,old woman,day history,HAS_SYMPTOM,present,A 44-year-old woman presented with a 3-day his...
1,E_DEP_002,day history,right flank,HAS_SYMPTOM,present,A 44-year-old woman presented with a 3-day his...
2,E_DEP_003,right flank,abdominal pain,HAS_SYMPTOM,present,A 44-year-old woman presented with a 3-day his...
3,E_DEP_004,old woman,day history,RELATED_TO,associate,A 44-year-old woman presented with a 3-day his...
4,E_DEP_005,day history,right flank,RELATED_TO,associate,A 44-year-old woman presented with a 3-day his...
5,E_DEP_006,right flank,abdominal pain,RELATED_TO,associate,A 44-year-old woman presented with a 3-day his...
6,E_DEP_007,computed tomography,cystic lesion,UNDERWENT_EXAM,undergo,She underwent contrast enhanced computed tomog...
7,E_DEP_008,computed tomography,cystic lesion,RELATED_TO,enhance,She underwent contrast enhanced computed tomog...
8,E_DEP_009,computed tomography,cystic lesion,SUPPORTS,demonstrate,She underwent contrast enhanced computed tomog...
9,E_DEP_010,pancreatic echotexture,internal septation,UNDERWENT_EXAM,undergo,"She subsequently underwent EUS-FNA, which reve..."


## Unir Entidades e Medidas Regex em um Grafo

In [5]:
# Unificação otimizada dos nós
final_nodes_df = pd.concat([nodes_df, measurements_df], ignore_index=True)
print(f"Total de Nós Unificados: {len(final_nodes_df)}")
display(final_nodes_df.head(10))

# Geração de arestas por proximidade de caracteres
combined_edges = []
edge_id_counter = 1

for _, ent_row in nodes_df.iterrows():
    for _, meas_row in measurements_df.iterrows():
        distancia = abs(ent_row['span_start'] - meas_row['span_start'])
        if distancia < 40:
            combined_edges.append({
                'edge_id': f"E_VAL_{edge_id_counter:03d}",
                'source_label': ent_row['label_normalizado'],
                'target_label': meas_row['label_normalizado'],
                'relation': 'HAS_VALUE',
                'character_distance': distancia
            })
            edge_id_counter += 1

final_edges_df = pd.DataFrame(combined_edges).drop_duplicates(subset=['source_label', 'target_label'])
print(f"\nTotal de Arestas de Valores geradas: {len(final_edges_df)}")
display(final_edges_df)

Total de Nós Unificados: 51


,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,MedicalConcept,old woman,old woman,5,7,10,19,None,None
1,MedicalConcept,day history,day history,12,14,39,50,None,None
2,MedicalConcept,right flank,right flank,15,17,54,65,None,None
3,MedicalConcept,abdominal pain,abdominal pain,20,22,85,99,None,None
4,MedicalConcept,medication history,medication history,34,36,170,188,None,None
5,MedicalConcept,physical examination,physical examination,43,45,229,249,None,None
6,MedicalConcept,computed tomography,computed tomography,52,54,300,319,None,None
7,MedicalConcept,cystic lesion,cystic lesion,59,61,341,354,None,None
8,MedicalConcept,pancreatic echotexture,pancreatic echotexture,86,88,472,494,None,None
9,MedicalConcept,internal septations,internal septation,101,103,543,562,None,None



Total de Arestas de Valores geradas: 13


,edge_id,source_label,target_label,relation,character_distance
0,E_VAL_001,computed tomography,6 cm,HAS_VALUE,37
1,E_VAL_002,cystic lesion,6 cm,HAS_VALUE,4
2,E_VAL_003,internal septation,6 cm,HAS_VALUE,27
3,E_VAL_004,internal septation,9 cm,HAS_VALUE,21
4,E_VAL_005,carbohydrate antigen,"12,476.5 ng/ml",HAS_VALUE,20
5,E_VAL_006,final pathology,9.5 cm,HAS_VALUE,27
6,E_VAL_007,final pathology,4.5 cm,HAS_VALUE,35
7,E_VAL_008,cm cyst,9.5 cm,HAS_VALUE,19
8,E_VAL_009,cm cyst,4.5 cm,HAS_VALUE,11
9,E_VAL_010,cm cyst,2.0 cm,HAS_VALUE,3


## Consolidar Todos os Nós e Arestas Finais

In [6]:
# Limpeza e unificação final das arestas sem redundâncias
edges_dep_clean = edges_dep_df[['edge_id', 'source_label', 'target_label', 'relation']].copy()
edges_val_clean = final_edges_df[['edge_id', 'source_label', 'target_label', 'relation']].copy()

final_graph_edges_df = pd.concat([edges_dep_clean, edges_val_clean], ignore_index=True)
final_graph_edges_df = final_graph_edges_df.drop_duplicates(subset=['source_label', 'target_label', 'relation']).reset_index(drop=True)

# Sequenciação limpa de IDs
final_graph_edges_df['edge_id'] = [f"E_{i+1:03d}" for i in range(len(final_graph_edges_df))]

print("=== GRAFO DE CONHECIMENTO CONSOLIDADO ===")
print(f"Total de Nós Finais: {len(final_nodes_df)}")
print(f"Total de Arestas Finais: {len(final_graph_edges_df)}")

display(final_graph_edges_df.head(15))

=== GRAFO DE CONHECIMENTO CONSOLIDADO ===
Total de Nós Finais: 51
Total de Arestas Finais: 54


,edge_id,source_label,target_label,relation
0,E_001,old woman,day history,HAS_SYMPTOM
1,E_002,day history,right flank,HAS_SYMPTOM
2,E_003,right flank,abdominal pain,HAS_SYMPTOM
3,E_004,old woman,day history,RELATED_TO
4,E_005,day history,right flank,RELATED_TO
5,E_006,right flank,abdominal pain,RELATED_TO
6,E_007,computed tomography,cystic lesion,UNDERWENT_EXAM
7,E_008,computed tomography,cystic lesion,RELATED_TO
8,E_009,computed tomography,cystic lesion,SUPPORTS
9,E_010,pancreatic echotexture,internal septation,UNDERWENT_EXAM
